# Sesión 4 · Transformación de datos — empleos (versión CON cola larga (outliers conservados))

**Curso:** Minería de Datos · UNIMINUTO Ibagué · 2026-2
**Ingreso:** salida de `3.0_preprocesamiento.ipynb` · **Paso (CRISP-DM):** Preparación de datos → *transformación*.

> Regla de oro de la sesión: **no se puede entrenar una IA con datos no numéricos**. Los valores categóricos, si se entregan tal cual, la sesgan. Aquí llevamos cada columna a una representación numérica *honesta* (one-hot u ordinal) y luego **escalamos** las numéricas para que ninguna columna "secuestre" las distancias que usarán KNN / K-means en sesiones posteriores.
>
> Este cuaderno es **transversal a las dos versiones** de la sesión 3. En la celda de configuración de la sección 0 se elige `con_outliers` (376.567 filas) o `sin_outliers` (341.090 filas). La sección 4 re-audita los outliers en tiempo de ejecución para decidir el escalador.

## 0 · Entorno, versión y carga

```text
Limpio -> Codificar categóricas -> Escalar -> (PCA si aplica) -> Modelar
```
Hoy cubrimos los tres primeros bloques.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.preprocessing import (
    KBinsDiscretizer, MinMaxScaler, RobustScaler, StandardScaler, OneHotEncoder,
)
from sklearn.decomposition import PCA

print("Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| sklearn", __import__("sklearn").__version__)

PROC = Path(r"C:\Users\ACER\Desktop\MineriaProtect\data\03_processed")

# <<< CONFIG: cambie el valor de VERSION para elegir la versión del dataset >>>
VERSION = "con_outliers"          # "con_outliers"  o  "sin_outliers"
CSV = PROC / ("empleos_limpio.csv" if VERSION == "con_outliers"
              else "empleos_limpio_sin_outliers.csv")
print("\nVersión:", VERSION, "| archivo:", CSV.name)

SAMPLE_SIZE = 50_000   # submuestra para las matrices densas de dummies (seccion 3)
RNG = np.random.RandomState(42)

Python 3.14.7 | pandas 3.0.2 | numpy 2.4.4 | sklearn 1.9.1

Versión: con_outliers | archivo: empleos_limpio.csv


In [2]:
df = pd.read_csv(
    CSV, dtype=str, encoding="utf-8", keep_default_na=False, na_values=[""]
)
print(f"dataset           : {df.shape[0]:,} filas x {df.shape[1]} columnas | "
      f"{df['resume_id'].nunique():,} personas")
print("\nNulos por columna (los NaN de ocupacion ya estan explicados por banderas en la sesion 3):")
print(df.isna().sum()[df.isna().sum() > 0].to_string())

dataset           : 376,567 filas x 14 columnas | 71,061 personas

Nulos por columna (los NaN de ocupacion ya estan explicados por banderas en la sesion 3):
occupation_code     28304
occupation_label    28304
isco_group          25805
isco_group_label    25805
isco_level          25805


---
## 1 · ¿Por qué cada columna es *one-hot* u *ordinal*?

La naturaleza de la variable decide la codificación. La regla del curso:

- **Nominales** (sin jerarquía natural) → **one-hot**: "aseo, libros, licores" no tienen orden, no se puede decir que uno "vale más" que otro. Numerarlos 1, 2, 3 inventaría distancia.
- **Ordinales** (jerarquía real y verificable) → **ordinal / KBins**: la duración de un empleo o el nivel educativo sí tienen orden ("jefe > auxiliar").

Las cifras de filas dependen de la versión elegida; todo se calcula en tiempo de ejecución.

| Columna | Naturaleza | Codificación elegida | Por qué |
|---|---|---|---|
| `start_date` / `end_date` | **Ordinal** (tiempo: Q1 1965 < Q1 2026) | ordinal (año×4+q) → `KBinsDiscretizer` (demo) + escalado | El tiempo tiene orden estricto y significado; jamás one-hot (≈206 categorías y se perdería el orden) |
| `university_level` | **Ordinal** (jerarquía de estudios) | mapeo ordinal manual 0–4 | "No reportado < Secondary school < Bachelor < Master < PhD" es jerarquía real; son solo 5 valores discretos, mapeo directo (KBins no aplica a 5 niveles fijos) |
| `isco_level` | **Ordinal por diseño** (nivel 1–4 ISCO) | → **se descarta** | En los datos es prácticamente **constante** (`'4'` en ~93% y `NaN` donde la ocupación es unknown). Una columna sin variabilidad no discrimina; su ausencia ya está capturada por `es_unknown_ocupacion` |
| `emparejado` | **Nominal** (ok / unknown / rescatado) | **one-hot** (`get_dummies`, 3 columnas) | Son 3 estados sin orden: "rescatado" no es mayor que "ok" |
| `matched_code` | **Nominal** (código con el que se emparejó) | **one-hot en esencia → se fusiona** | Coincide 100% con `occupation_code` cuando `emparejado='ok'` y guarda el código recuperado en `rescatado`; se fusiona en `occupation_code` (ver celda abajo) |
| `occupation_code` | **Nominal** (≈2.856 códigos) | **one-hot** (`get_dummies`) | Códigos de ocupación ISCO son etiquetas sin orden. Ojo: son 2.856 categorías → *explosión de columnas* (se analiza abajo) |
| `occupation_label` | **Nominal** (nombre del cargo) | → **se descarta** | Es 1:1 con `occupation_code` (una etiqueta por código): duplicaría las 2.856 columnas sin información nueva |
| `isco_group` / `isco_group_label` | **Nominal** (grupo ISCO 4 dígitos) | → **se descartan** | `isco_group` es un *agregado* del código de ocupación (sus 4 primeros dígitos) y `isco_group_label` es 1:1 con él; uno-hotear `occupation_code` ya los contiene |
| `es_unknown_ocupacion`, `es_rescatado`, `es_vigente` | **Binarias** (flags True/False) | 0/1 directo (equivale a one-hot con `drop='first'`) | Ya son binarias: escalarlas no aporta y rompe su legibilidad (slide 21) |
| `resume_id` | Identificador | **se descarta** | Es la clave primaria (una por persona), no un patrón |

### 1.1 Verificación de los descartes (auditoría de redundancia)

Los descartes NO son opinión: se demuestran con los datos.


In [3]:
ok = df[df["emparejado"] == "ok"]
print("matched_code == occupation_code cuando emparejado='ok':",
      round(float((ok["matched_code"] == ok["occupation_code"]).mean()), 4))

g = df.dropna(subset=["occupation_code", "occupation_label"])
print("occupation_label x code (max etiquetas por codigo, 1 = 1:1):",
      int(g.groupby("occupation_code")["occupation_label"].nunique().max()))

g2 = df.dropna(subset=["isco_group", "isco_group_label"])
print("isco_group_label x group (max etiquetas por grupo, 1 = 1:1):",
      int(g2.groupby("isco_group")["isco_group_label"].nunique().max()))

print("isco_level valores distintos:", df["isco_level"].dropna().unique().tolist())
print("NaN de isco_level == es_unknown_ocupacion=True:",
      int((df["isco_level"].isna() == (df["es_unknown_ocupacion"] == "True")).all()))

print("\nmatched_code en emparejado='unknown':",
      df.loc[df["emparejado"] == "unknown", "matched_code"].unique()[:3])
print("matched_code en emparejado='rescatado' (codigos recuperados):",
      df.loc[df["emparejado"] == "rescatado", "matched_code"].dropna().unique()[:3])

matched_code == occupation_code cuando emparejado='ok': 1.0


occupation_label x code (max etiquetas por codigo, 1 = 1:1): 1


isco_group_label x group (max etiquetas por grupo, 1 = 1:1): 1
isco_level valores distintos: ['4']
NaN de isco_level == es_unknown_ocupacion=True: 1

matched_code en emparejado='unknown': <ArrowStringArray>
['unknown']
Length: 1, dtype: str
matched_code en emparejado='rescatado' (codigos recuperados): <ArrowStringArray>
['2659.2.2', '3412.4.9', '2424.3']
Length: 3, dtype: str


### 1.2 Fusión: `occupation_code` conserva el código recuperado de `rescatado`

Para `rescatado` el código vive en `matched_code`; para `unknown` ambas están vacías/`'unknown'`. Fusionamos para dejar **una sola** columna de ocupación.


In [4]:
ocup = df["occupation_code"].copy()
fusion = ocup.isna() & (df["matched_code"] != "unknown")
ocup.loc[fusion] = df.loc[fusion, "matched_code"]      # rescatado: trae el codigo recuperado

print("NaN restantes de ocupacion tras fusion:", int(ocup.isna().sum()),
      "| deben coincidir con emparejado='unknown':",
      int((ocup.isna() == (df["emparejado"] == "unknown")).all()))

df = df.assign(occupation_code=ocup)

NaN restantes de ocupacion tras fusion: 25805 | deben coincidir con emparejado='unknown': 1


---
## 2 · Codificación ordinal

### 2.1 Fechas → ordinal numérico

`Q1 2011` es texto. Lo llevamos a `año × 4 + trimestre` (Q1 2011 = 2011×4+1 = 8.045), que preserva el orden temporal y permite operaciones aritméticas.


In [5]:
RE = r"^Q([1-4])\s+(\d{4})$"

def to_ordinal(x):
    p = x.str.extract(RE)
    return (pd.to_numeric(p[1], errors="coerce") * 4 +
            pd.to_numeric(p[0], errors="coerce"))

start_ord = to_ordinal(df["start_date"])
end_ord   = to_ordinal(df["end_date"])

# 'Present' = censura (el empleo sigue). Lo ubicamos en el trimestre del proyecto (Q1 2026).
presentes = df["end_date"] == "Present"
end_ord = end_ord.mask(presentes, 2026 * 4 + 1)

# Duracion trimestral (feature comportamental derivada de las dos fechas)
dur_q = end_ord - start_ord

print("start_ord : min", start_ord.min(), "max", start_ord.max())
print("end_ord   : min", end_ord.min(), "max", end_ord.max(),
      "| 'Present' reubicados:", int(presentes.sum()))
print("dur_q     : min", dur_q.min(), "max", dur_q.max())

df = df.assign(start_ord=start_ord, end_ord=end_ord, dur_q=dur_q)

start_ord : min 7861 max 8085
end_ord   : min 7881.0 max 8105.0 | 'Present' reubicados: 18956
dur_q     : min 1.0 max 212.0


### 2.2 `university_level` → mapeo ordinal manual

Son **5 niveles discretos con jerarquía real**; un mapeo manual explícito es más fiel que discretizar (KBins agruparía niveles arbitrariamente). Documentamos la escala en el propio código.


In [6]:
ORDEN_UNIV = {
    "No reportado":     0,   # ausencia como estado, no como 0 engañoso de estudios
    "Secondary school": 1,
    "Bachelor":         2,
    "Master":           3,
    "PhD":              4,
}
univ_ord = df["university_level"].map(ORDEN_UNIV)
print("univ_ord unicos:", sorted(univ_ord.unique().tolist()))
print("orden verificado:", [u for u in sorted(univ_ord.unique())])

df = df.assign(univ_ord=univ_ord)

univ_ord unicos: [0, 1, 2, 3, 4]
orden verificado: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


### 2.3 `isco_level`: por qué se descarta (es constante)

Es **ordinal por diseño** (nivel de competencia ISCO 1–4), pero en estos datos no discrimina:
- ~93% de las filas vale `4`; el resto es `NaN` = ocupación desconocida.
- El `NaN` ya lo explica la bandera `es_unknown_ocupacion`.

Dejarlo como número casi constante no aporta información al modelo y genera una falsa sensación de variable.


In [7]:
isco_lv = df["isco_level"]
tabla = isco_lv.fillna("NaN").value_counts()
print("Conteo de isco_level:")
print(tabla.to_string())
prop = tabla.get("4", 0) / len(df)
print(f"\nproporcion de filas == '4': {prop:.4f}  -> columna ~constante, se descarta.")

Conteo de isco_level:
isco_level
4      350762
NaN     25805

proporcion de filas == '4': 0.9315  -> columna ~constante, se descarta.


### 2.4 `KBinsDiscretizer` (método del curso) aplicado a las fechas

El curso lo usa para convertir una continua en **segmentos ordinales** de igual tamaño (*quantile* = ~misma cantidad de observaciones por rango). Lo aplicamos a `start_ord` y `end_ord`.


In [8]:
kbd_inicio = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")
kbd_fin    = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")

bins_inicio = kbd_inicio.fit_transform(start_ord.to_frame()).ravel()
bins_fin    = kbd_fin.fit_transform(end_ord.to_frame()).ravel()

print("Bordes de start_ord:", kbd_inicio.bin_edges_[0].round(0))
print("Bordes de end_ord  :", kbd_fin.bin_edges_[0].round(0))
print("\nStart en 5 segmentos (mismo tamano por cuantiles):")
print(pd.Series(bins_inicio).value_counts().sort_index().to_string())

Bordes de start_ord: [7861. 8013. 8037. 8049. 8060. 8085.]
Bordes de end_ord  : [7884. 8029. 8045. 8057. 8067. 8105.]

Start en 5 segmentos (mismo tamano por cuantiles):
0.0    69820
1.0    77191
2.0    72797
3.0    79867
4.0    76892


---
## 3 · Codificación one-hot con `get_dummies`

### 3.1 El problema de las ~2.856 ocupaciones

> *"ciudad × país explota"* — el curso advierte que pocas columnas nominales de alta cardinalidad generan muchísimas columnas. Una ocupación por persona → filas enormemente dispersas.

El grueso de las filas usa ocupaciones únicas. Por eso:

- La demostración con `get_dummies` (matriz densa) se hace sobre una **submuestra representativa de 50.000 filas**; se mantiene la semilla para reproducibilidad.
- Al final de la sección se muestra la alternativa de **producción**: `OneHotEncoder(sparse_output=True)`, que codifica el dataset completo sin explotar la memoria.

Primero, el **bloque de frecuencias**:


In [9]:
frec = df["occupation_code"].dropna().value_counts()
print("Categorias unicas (con datos):", int(frec.size), "-> ~2.800 columnas dummy\n")
print(frec.head(10).to_string())

idx = RNG.choice(len(df), size=min(SAMPLE_SIZE, len(df)), replace=False)
df_s = df.iloc[idx].reset_index(drop=True)
print(f"\nSubmuestra de trabajo: {len(df_s):,} filas")

Categorias unicas (con datos): 2872 -> ~2.800 columnas dummy

occupation_code
3343.1      16864
5223.4      10017
5223.6       9676
9333.8       7106
5230.1       5990
9412.1       5747
5131.2       5347
9112.2       4785
9329.1       4654
3343.1.7     3675

Submuestra de trabajo: 50,000 filas


In [10]:
# one-hot nominal
dummies_emparejado = pd.get_dummies(df_s["emparejado"], prefix="emparejado", dtype=np.uint8)

# one-hot ocupacion (NaN = sin_ocupacion queda en todo 0 y es explicada por la bandera)
dummies_ocupacion  = pd.get_dummies(df_s["occupation_code"], prefix="occupation", dtype=np.uint8)

# flags binarias -> 0/1 (ya son one-hot de k-1, no requieren get_dummies ni escalado)
flags = (df_s[["es_unknown_ocupacion", "es_rescatado", "es_vigente"]]
         .replace({"True": 1, "False": 0})
         .astype(np.uint8))

print("emparejado ->", dummies_emparejado.shape[1], "columnas",
      dummies_emparejado.columns.tolist())
print("ocupacion  ->", dummies_ocupacion.shape[1], "columnas")
print("flags      ->", flags.shape[1], "columnas", flags.columns.tolist())

emparejado -> 3 columnas ['emparejado_ok', 'emparejado_rescatado', 'emparejado_unknown']
ocupacion  -> 2312 columnas
flags      -> 3 columnas ['es_unknown_ocupacion', 'es_rescatado', 'es_vigente']


> **Recuerda (slides 18–21 de la sesión 4):** las columnas dummy y binarias **NO se escalan**. Escalarlas las vuelve ilegibles sin ganar nada.

### 3.2 Alternativa de producción: `OneHotEncoder` con salida dispersa (dataset completo)

Mismo resultado, pero usa una matriz *sparse* (guarda solo los 1). Con esto **sí** se puede codificar el dataset completo sin 1–9 GB en RAM.


In [11]:
ohe = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X_ohe_full = ohe.fit_transform(df[["emparejado", "occupation_code"]])
print("OneHotEncoder (full, sparse):", X_ohe_full.shape)
print("Memoria relativa del bloque cf. densa:", X_ohe_full.data.nbytes, "bytes aprx.")

OneHotEncoder (full, sparse): (376567, 2876)
Memoria relativa del bloque cf. densa: 6025072 bytes aprx.


---
## 4 · Escalamiento: ¿cuál de los 3 y por qué?

### 4.1 Los outliers documentados en la sesión 3 (auditoría en tiempo de ejecución)

La sesión 3 trató cada versión de forma distinta: en la versión **con** cola larga conservó los extremos (decisión de negocio); en la versión **sin** cola larga los eliminó (`dur ≥ 25` trimestres). Aquí se re-calculan los límites Tukey (IQR) **sobre la versión elegida**, en las dos variantes de `dur_q` (con `Present` → censura, igual que la sesión 3, y con `Present` → Q1 2026, nuestro encoding):


In [12]:
def tukey(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return q1, q3, iqr, li, ls, int(((s < li) | (s > ls)).sum())

dur_q_ref = (to_ordinal(df["end_date"]) - start_ord)   # Present -> NaN como sesion 3

filas = []
for nombre, var in [("start_ord", start_ord), ("end_ord", end_ord),
                    ("dur_q (Present censurado)", dur_q_ref),
                    ("dur_q (Present = Q1 2026)", dur_q),
                    ("university_map", univ_ord)]:
    q1, q3, iqr, li, ls, out = tukey(var.dropna())
    filas.append([nombre, round(q1, 1), round(q3, 1), round(iqr, 1),
                  round(li, 1), round(ls, 1), out, round(out / len(df), 3)])

pd.DataFrame(filas, columns=["variable", "Q1", "Q3", "IQR", "lim_inf",
                             "lim_sup", "outliers", "prop_outliers"])

,variable,Q1,Q3,IQR,lim_inf,lim_sup,outliers,prop_outliers
0,start_ord,8021.0,8057.0,36.0,7967.0,8111.0,20962,0.056
1,end_ord,8033.0,8065.0,32.0,7985.0,8113.0,17993,0.048
2,dur_q (Present censurado),1.0,10.0,9.0,-12.5,23.5,35477,0.094
3,dur_q (Present = Q1 2026),1.0,12.0,11.0,-15.5,28.5,41410,0.110
4,university_map,1.0,2.0,1.0,-0.5,3.5,1652,0.004


### 4.2 Decisiones de escalador (una por columna)

| Columna | Outliers detectados | Escalador | Por qué |
|---|---|---|---|
| `start_ord` | ~21 mil (cola baja: empleos iniciados en los 60s–70s) | **RobustScaler** | La mediana y el IQR no se dejan arrastrar por la cola baja; conserva la estructura del calendario |
| `end_ord` | ~18 mil (cola baja similar, más la censura de `Present`) | **RobustScaler** | Idem |
| `dur_q` | cola larga real (~41 mil en nuestro encoding; 35.477 según la sesión 3 con censura) | **RobustScaler** | Es el fenómeno documentado en la sesión 3: empleos de décadas. Robust lo "aísla" |
| `university_map` | rango fijo [0, 4] por diseño, sin atípicos | **MinMaxScaler** | Rango acotado y conocido → el caso natural del MinMax; lleva todo a [0, 1] |
| `StandardScaler` | — | *no se elige* | Usa media/desviación: con la cola larga/las fechas viejas la media se corrompe (sensibilidad moderada). Se muestra la comparación abajo |

> **Fuga de datos (slide 21 · Error 1):** el escalador se ajusta (`fit`) **solo** sobre la muestra de trabajo; jamás sobre los datos de prueba. Aquí lo hacemos explícito.


In [13]:
# 1) Robust para fechas y duracion
rob = RobustScaler()
escaladas_rob = pd.DataFrame(
    rob.fit_transform(df_s[["start_ord", "end_ord", "dur_q"]].to_numpy()),
    columns=["start_ord_rob", "end_ord_rob", "dur_q_rob"], index=df_s.index)

# 2) MinMax para education (rango acotado sin atipicos)
mm = MinMaxScaler()
univ_mm = pd.DataFrame(mm.fit_transform(df_s[["univ_ord"]].to_numpy()),
                       columns=["university_mm"], index=df_s.index)

print("Robust -> mediana despues de escalar (debe ser ~0):",
      escaladas_rob.median().round(4).tolist())
print("MinMax -> min/max despues de escalar (deben ser 0/1):",
      (univ_mm.min().round(4).values[0], univ_mm.max().round(4).values[0]))

# 3) Comparacion de los tres escaladores sobre dur_q (10 observaciones)
durq_demo = df_s["dur_q"].to_numpy().reshape(-1, 1).astype(float)
tabla = pd.DataFrame({
    "crudo":      durq_demo[:, 0],
    "minmax":     MinMaxScaler().fit_transform(durq_demo)[:, 0],
    "standard":   StandardScaler().fit_transform(durq_demo)[:, 0],
    "robust":     RobustScaler().fit_transform(durq_demo)[:, 0],
})
print("\nEscaladores sobre dur_q (10 primeras de la muestra):")
print(tabla.head(10).round(3).to_string())
print("\nEl robust mantiene la estructura de la distribucion; el minmax y el")
print("standard quedan a merced de los extremos (cola).")

Robust -> mediana despues de escalar (debe ser ~0): [0.0, 0.0, 0.0]
MinMax -> min/max despues de escalar (deben ser 0/1): (np.float64(0.0), np.float64(1.0))

Escaladores sobre dur_q (10 primeras de la muestra):
   crudo  minmax  standard  robust
0    1.0   0.000    -0.571  -0.273
1    1.0   0.000    -0.571  -0.273
2    1.0   0.000    -0.571  -0.273
3    1.0   0.000    -0.571  -0.273
4    1.0   0.000    -0.571  -0.273
5    1.0   0.000    -0.571  -0.273
6    4.0   0.015    -0.407   0.000
7    4.0   0.015    -0.407   0.000
8    4.0   0.015    -0.407   0.000
9    4.0   0.015    -0.407   0.000

El robust mantiene la estructura de la distribucion; el minmax y el
standard quedan a merced de los extremos (cola).


### 4.3 ¿PCA? Auditoría de correlación (slides 12–14)

El pipeline de la sesión 4 cierra con **"PCA, si aplica"**: la clase pide una **correlación** antes de decidir (`pca si o no aplica hay que hacer una correlacion`). PCA busca pocas direcciones (componentes) que expliquen la mayor varianza de columnas **numéricas escaladas**; si las columnas están poco correlacionadas, no hay direcciones que comprimir y PCA **no aplica**.

Aquí lo auditamos con honestidad:

1. **Escalamos** las numéricas con `StandardScaler` (el requisito del slide 12 es PCA sobre datos escalados; caso donde Standard **sí** corresponde).
2. **Correlaciones** de Pearson entre las numéricas.
3. **Varianza acumulada** de las componentes: necesitamos ~80–90 % para justificar comprimir.
4. Las **dummies** (one-hot) quedan fuera: son categóricas puras y ya se "comprimió" convirtiendo categorías en columnas; PCA sobre 2.800 dummies araña información estructural (slide 14).


In [14]:
# 1) Matriz de correlacion de las numericas de la muestra
X_num = df_s[["start_ord", "end_ord", "dur_q", "univ_ord"]].astype(float)
print("Correlacion de Pearson (numericas de la muestra):")
print(X_num.corr().round(3).to_string())
print()
print("Nota: dur_q = end_ord - start_ord + 1  ->  tres de las cuatro columnas")
print("guardan una relacion lineal casi exacta (dur_q = end_ord - start_ord + 1).")

# 2) PCA sobre las numericas estandarizadas (StandardScaler = prerequisito del slide 12)
Xp_std = StandardScaler().fit_transform(X_num)
pca = PCA(random_state=RNG)
Xp = pca.fit_transform(Xp_std)
var_ac = np.cumsum(pca.explained_variance_ratio_)
df_var = pd.DataFrame({
    "componente": range(1, 5),
    "varianza_individual": np.round(pca.explained_variance_ratio_[:4], 4),
    "varianza_acumulada": np.round(var_ac[:4], 4)},
).set_index("componente")
print("\nVarianza explicada del PCA sobre las 4 numericas:")
print(df_var.to_string())
print(f"\nPara llegar a ~80% se necesitan {int((var_ac < 0.80).sum()) + 1} de 4 componentes")

# 3) ?Que pasa con las dummies? Solo contamos columnas: no entran a PCA
print("\nColumnas one-hot de occupation_code:", dummies_ocupacion.shape[1], "-> no entran a PCA")
print("Diagnostico: PCA casi no comprime aqui. No se aplica a X final.")

Correlacion de Pearson (numericas de la muestra):
           start_ord  end_ord  dur_q  univ_ord
start_ord      1.000    0.836 -0.366     0.112
end_ord        0.836    1.000  0.204     0.090
dur_q         -0.366    0.204  1.000    -0.047
univ_ord       0.112    0.090 -0.047     1.000

Nota: dur_q = end_ord - start_ord + 1  ->  tres de las cuatro columnas
guardan una relacion lineal casi exacta (dur_q = end_ord - start_ord + 1).

Varianza explicada del PCA sobre las 4 numericas:
            varianza_individual  varianza_acumulada
componente                                         
1                        0.4697              0.4697
2                        0.2876              0.7573
3                        0.2427              1.0000
4                        0.0000              1.0000

Para llegar a ~80% se necesitan 3 de 4 componentes

Columnas one-hot de occupation_code: 2312 -> no entran a PCA
Diagnostico: PCA casi no comprime aqui. No se aplica a X final.


**Conclusión de la auditoría (misma lógica del resultado de TelecomUNO en el slide 13):** PCA **no comprime** en este dataset. Las fechas/duración son una relación lineal (dependencia casi exacta entre `dur_q`, `start_ord`, `end_ord`) → la 4.ª componente recoge ≈ 0 % de varianza, y aun así se necesitan casi todas las componentes para 80 %. `univ_ord` es casi independiente de las demás → suma su propia dimensión. `PCA` se **descartaría** aun si siguiéramos adelante con sólo numéricas, y por diseño queda excluido de la matriz final (dummies inmensas + columna densa pequeña). Su valor aquí fue **diagnóstico**: confirmó que no hay una estructura de varianza compartida que comprimir sin romper el one-hot.


---
## 5 · Matriz X final lista para modelar

**Formato (espejo de la sesión 4, slide 16):** numéricas/ordinales escaladas + dummies + banderas. Nada de texto, nada de binarias escaladas, sin `resume_id`, sin columnas descartadas.


In [15]:
X_final = pd.concat([
    escaladas_rob,                     # start_ord_rob, end_ord_rob, dur_q_rob
    univ_mm,                           # university_mm
    dummies_emparejado,                # emparejado_ok / _unknown / _rescatado
    dummies_ocupacion,                 # occupation_<codigo>
    flags,                             # es_unknown_ocupacion, es_rescatado, es_vigente
], axis=1)

print("X final (submuestra):", X_final.shape)
print("\nPrimeras columnas:", X_final.columns[:12].tolist(), "...")
print("\nComprobaciones de sanidad:")
print(" - dtypes no-numericos:", int(X_final.select_dtypes(exclude=["number"]).shape[1]))
print(" - nulos:", int(X_final.isna().sum().sum()))
X_final.head(3)

X final (submuestra): (50000, 2322)

Primeras columnas: ['start_ord_rob', 'end_ord_rob', 'dur_q_rob', 'university_mm', 'emparejado_ok', 'emparejado_rescatado', 'emparejado_unknown', 'occupation_0110.10', 'occupation_0110.11', 'occupation_0110.12', 'occupation_0110.2', 'occupation_0110.3'] ...

Comprobaciones de sanidad:
 - dtypes no-numericos: 0


 - nulos: 0


,start_ord_rob,end_ord_rob,dur_q_rob,university_mm,emparejado_ok,emparejado_rescatado,emparejado_unknown,occupation_0110.10,occupation_0110.11,occupation_0110.12,...,occupation_9629.1,occupation_9629.2,occupation_9629.3,occupation_9629.4,occupation_9629.5,occupation_9629.6,occupation_9629.7,es_unknown_ocupacion,es_rescatado,es_vigente
0,1.027778,0.93750,-0.272727,0.50,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0.194444,0.00000,-0.272727,0.25,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0.555556,0.40625,-0.272727,0.75,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


> La variable objetivo `y` aún no existe: la minería de **transiciones laborales** (qué ocupaciones siguen a cuáles por persona) se formaliza en la siguiente fase. Por ahora entregamos la **X** que habilitará KNN / K-means / árboles en sesiones 7–9.

### Guardar (opcional)

Descomente la siguiente celda si quiere persistir la matriz. Nota: 50.000×~2.800 columnas densas ocupan ~100–150 MB; para el dataset completo use el bloque `OneHotEncoder` sparse.


In [16]:
# X_final.to_csv(PROC / f"X_empleos_{VERSION}_transformado_sample.csv", index=False)
# print("Guardado:", PROC / f"X_empleos_{VERSION}_transformado_sample.csv")

---
## 6 · Resumen del pipeline y cierre

```text
empleos_limpio.csv (376.567 (con cola larga) x 14)
  ├─ 1  Ordinal:  fechas        -> anio*4+q -> KBinsDiscretizer (demo) + RobustScaler
  ├─ 2  Ordinal:  university_level -> mapeo manual 0-4 -> MinMaxScaler
  ├─ 3  One-hot:  emparejado (3) + occupation_code (~2.856)  [get_dummies demo / OHE-sparse prod]
  ├─ 4  Binarias: es_unknown_ocupacion, es_rescatado, es_vigente (0/1, sin escalar)
  ├─ 5  Derivada: dur_q (duracion trimestral) -> RobustScaler
  └─ -  Descartadas (justificado): resume_id, matched_code (fusionado), occupation_label
      (1:1), isco_group / isco_group_label (agregados), isco_level (constante)
```

**Decisiones clave de esta sesión**

1. `isco_level` es ordinal por diseño pero **constante** en los datos → descartada (su `NaN` = `es_unknown_ocupacion`).
2. Redundancias demostradas → `matched_code`, `occupation_label`, `isco_group*` descartados tras auditoría (1:1 / prefijo / fusión).
3. Escaladores elegidos **por los outliers re-auditados en la sección 4.1**: robust para fechas y duración (colas reales), **minmax** para educación (rango acotado sin atípicos) y standard **descartado** por sensibilidad a las colas.
4. Dummies y banderas binarias **no se escalan** (slide 21).
5. `fit` del escalador solo sobre la muestra de trabajo (evita fuga de datos).
6. **PCA auditado y descartado (sección 4.3):** las numéricas son ~3 dimensiones lineales (dependencia `dur_q` = `end − start` + 1) y `univ_ord` es independiente → no hay varianza compartida que comprimir; las dummies no entran a PCA por diseño. No se aplica a X final.

**Comparación con la otra versión (4.0 / 4.1):** el pipeline es idéntico; lo que cambia es la magnitud de los outliers y, por tanto, la justificación del robust en `dur_q`.

**Siguiente paso:** sesión 5 (EDA: histogramas, correlaciones, mapa de calor) sobre esta X y las banderas.
